In [1]:
from datetime import datetime
from itertools import chain
import pandas as pd
from bs4 import BeautifulSoup
from helpers import get_request

In [2]:
def get_pages(num_pages: int):
    for i in range(num_pages):
        if i % 10 == 0:
            print("Getting page", i, "/", num_pages)

        try:
            response = get_request(f"https://www.ufc.com/trending/all?page={i}")
            yield response.text
        except Exception as e:
            print("Error getting page", i)
            print(e)

            yield None

In [3]:
# I ran this on 16 Sept. 2025 with num_pages_to_get = 200, 
# and the oldest page I got was from 12 Jan. 2025.
# Adjust accordingly for the date range you need
num_pages_to_get = 10
pages = [
    page for page in get_pages(num_pages_to_get)
    if page is not None
]
print("Got", len(pages), "pages")

Getting page 0 / 10
Got 10 pages


In [4]:
def get_articles_data(page_content: str):
    soup = BeautifulSoup(page_content, "html.parser")
    articles = soup.find_all("a", "c-card--grid-card-trending")

    articles_data = [
        {
            "type": link.find_all("span", "c-card--grid-card-trending__info-prefix")[0].text,
            "time_ago": link.find_all("div", "field--name-datetime")[0].text,
            "headline": link.find_all("h3", "c-card--grid-card-trending__headline")[0].text.strip(),
            "link": link["href"]
        }
        for link in articles
    ]

    for data in articles_data:
        yield data

In [5]:
flattened_articles = list(chain.from_iterable(get_articles_data(page) for page in pages))
flattened_articles

[{'type': 'Athletes',
  'time_ago': '4 hours ago',
  'headline': 'Gabriel Bonfim | Leveled Up By A Legend',
  'link': '/news/gabriel-bonfim-leveled-up-legend-ufc-vegas-111'},
 {'type': 'Athletes',
  'time_ago': '5 hours ago',
  'headline': 'Chris Padilla | Blocking Out The Noise',
  'link': '/news/chris-padilla-blocking-out-the-noise-ufc-fight-night-bonfim-vs-brown'},
 {'type': 'Athletes',
  'time_ago': '5 hours ago',
  'headline': 'Matt Schnell | Honest, Excited, And Dangerous',
  'link': '/news/matt-schnell-honest-excited-and-dangerous-ufc-fight-night-bonfim-vs-brown'},
 {'type': 'Interviews',
  'time_ago': '5 hours ago',
  'headline': 'Zachary Reese Fight Week Interview | UFC Fight Night…',
  'link': '/video/152387'},
 {'type': 'Athletes',
  'time_ago': '5 hours ago',
  'headline': 'Tecia Pennington | At A Crossroad',
  'link': '/news/tecia-pennington-crossroad-ufc-fight-night-bonfim-vs-brown'},
 {'type': 'Announcements',
  'time_ago': '6 hours ago',
  'headline': 'Updates To UFC Fi

In [6]:
articles_df = pd.DataFrame(data=flattened_articles)
articles_df

,type,time_ago,headline,link
0,Athletes,4 hours ago,Gabriel Bonfim | Leveled Up By A Legend,/news/gabriel-bonfim-leveled-up-legend-ufc-veg...
1,Athletes,5 hours ago,Chris Padilla | Blocking Out The Noise,/news/chris-padilla-blocking-out-the-noise-ufc...
2,Athletes,5 hours ago,"Matt Schnell | Honest, Excited, And Dangerous",/news/matt-schnell-honest-excited-and-dangerou...
3,Interviews,5 hours ago,Zachary Reese Fight Week Interview | UFC Fight...,/video/152387
4,Athletes,5 hours ago,Tecia Pennington | At A Crossroad,/news/tecia-pennington-crossroad-ufc-fight-nig...
...,...,...,...,...
134,Interviews,1 week ago,Mackenzie Dern Talks With John Gooden | UFC 321,/video/152070
135,Interviews,1 week ago,Ciryl Gane Talks With John Gooden | UFC 321,/video/152069
136,Interviews,1 week ago,Tom Aspinall Talks With John Gooden | UFC 321,/video/152067
137,Athletes,1 week ago,Seriously: Do Not Blink When Tom Aspinall And ...,/news/seriously-do-not-blink-when-tom-aspinall...


In [7]:
filename = f"./article_links_{datetime.now().strftime('%d-%m-%y_%H-%M-%S')}.csv"
articles_df\
    .set_index("link")\
    .to_csv(filename)

print("Saved article links to", filename)

Saved article links to ./article_links_06-11-25_20-40-38.csv
